<a href="https://colab.research.google.com/github/yashasnot/AI-Document-Intelligence/blob/main/Routing_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Load the M3 routing dataset
FILE_NAME = "unified_routing 2.csv"

df = pd.read_csv(FILE_NAME)

print("✅ M3 Route Agent dataset loaded")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\nColumns:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

✅ M3 Route Agent dataset loaded
Rows: 2037
Columns: 23

Columns:
1. zone_id
2. region
3. latitude
4. longitude
5. pfz_distance_km
6. pfz_depth_m
7. water_depth_m
8. current_u_ms
9. current_v_ms
10. current_speed_ms
11. eez_boundary_distance_km
12. nearest_boundary_type
13. geofence_caution
14. inside_mpa
15. mpa_distance_km
16. mpa_caution
17. distance_km
18. travel_time_hr
19. fuel_l_proxy
20. route_score
21. routing_class
22. agent
23. data_region


In [2]:
# STEP 2 — Route Dataset Preprocessing

# Keep a copy of raw data
route_df = df.copy()

# Convert important numeric columns to numeric
numeric_cols = [
    "latitude",
    "longitude",
    "pfz_distance_km",
    "pfz_depth_m",
    "water_depth_m",
    "current_u_ms",
    "current_v_ms",
    "current_speed_ms",
    "eez_boundary_distance_km",
    "mpa_distance_km",
    "distance_km",
    "travel_time_hr",
    "fuel_l_proxy",
    "route_score"
]

for col in numeric_cols:
    if col in route_df.columns:
        route_df[col] = pd.to_numeric(route_df[col], errors="coerce")

# Remove duplicate zone IDs
route_df = route_df.drop_duplicates(subset=["zone_id"]).reset_index(drop=True)

# Validate coordinates
route_df = route_df[
    route_df["latitude"].between(-90, 90)
    & route_df["longitude"].between(-180, 180)
].reset_index(drop=True)

print("✅ Preprocessing completed")
print("Shape after preprocessing:", route_df.shape)

print("\nMissing values:")
print(route_df.isnull().sum())

print("\nFirst 5 rows:")
display(route_df.head())

✅ Preprocessing completed
Shape after preprocessing: (2037, 23)

Missing values:
zone_id                        0
region                         0
latitude                       0
longitude                      0
pfz_distance_km                0
pfz_depth_m                 1700
water_depth_m                  0
current_u_ms                 495
current_v_ms                 495
current_speed_ms             495
eez_boundary_distance_km       0
nearest_boundary_type          0
geofence_caution               0
inside_mpa                     0
mpa_distance_km                0
mpa_caution                    0
distance_km                    0
travel_time_hr                 0
fuel_l_proxy                   0
route_score                  495
routing_class                495
agent                          0
data_region                    0
dtype: int64

First 5 rows:


,zone_id,region,latitude,longitude,pfz_distance_km,pfz_depth_m,water_depth_m,current_u_ms,current_v_ms,current_speed_ms,...,inside_mpa,mpa_distance_km,mpa_caution,distance_km,travel_time_hr,fuel_l_proxy,route_score,routing_class,agent,data_region
0,PFZ0001,ANDAMAN,13.170278,92.640278,20.0,15.0,20.0,NaN,NaN,NaN,...,0,1053.068365,0,20.0,1.111111,8.888889,NaN,NaN,ROUTE_AGENT,BROADER_INDIAN_MARINE_REGION
1,PFZ0002,ANDAMAN,11.760000,92.946944,16.0,97.0,102.0,NaN,NaN,NaN,...,0,1213.398008,0,16.0,0.888889,7.111111,NaN,NaN,ROUTE_AGENT,BROADER_INDIAN_MARINE_REGION
2,PFZ0003,ANDAMAN,11.745000,93.004444,8.0,67.0,75.0,NaN,NaN,NaN,...,0,1215.136296,0,8.0,0.444444,3.555556,NaN,NaN,ROUTE_AGENT,BROADER_INDIAN_MARINE_REGION
3,PFZ0004,ANDAMAN,11.694444,93.021944,27.0,226.0,249.0,NaN,NaN,NaN,...,0,1220.900345,0,27.0,1.500000,12.000000,NaN,NaN,ROUTE_AGENT,BROADER_INDIAN_MARINE_REGION
4,PFZ0005,ANDAMAN,11.632500,93.021111,33.0,254.0,230.0,NaN,NaN,NaN,...,0,1227.940194,0,33.0,1.833333,14.666667,NaN,NaN,ROUTE_AGENT,BROADER_INDIAN_MARINE_REGION


In [3]:
# STEP 3 — Prepare Route Agent Variables

route_features = [
    "zone_id",
    "region",
    "latitude",
    "longitude",
    "pfz_distance_km",
    "pfz_depth_m",
    "water_depth_m",
    "current_u_ms",
    "current_v_ms",
    "current_speed_ms",
    "eez_boundary_distance_km",
    "nearest_boundary_type",
    "geofence_caution",
    "inside_mpa",
    "mpa_distance_km",
    "mpa_caution",
    "distance_km",
    "travel_time_hr",
    "fuel_l_proxy"
]

# Keep only variables that exist in the dataset
available_features = [
    col for col in route_features
    if col in route_df.columns
]

routing_data = route_df[available_features].copy()

print("✅ Route variables prepared")

print("\nRouting variables:")
print(routing_data.columns.tolist())

print("\nRouting data shape:")
print(routing_data.shape)

print("\nSample:")
display(routing_data.head())

✅ Route variables prepared

Routing variables:
['zone_id', 'region', 'latitude', 'longitude', 'pfz_distance_km', 'pfz_depth_m', 'water_depth_m', 'current_u_ms', 'current_v_ms', 'current_speed_ms', 'eez_boundary_distance_km', 'nearest_boundary_type', 'geofence_caution', 'inside_mpa', 'mpa_distance_km', 'mpa_caution', 'distance_km', 'travel_time_hr', 'fuel_l_proxy']

Routing data shape:
(2037, 19)

Sample:


,zone_id,region,latitude,longitude,pfz_distance_km,pfz_depth_m,water_depth_m,current_u_ms,current_v_ms,current_speed_ms,eez_boundary_distance_km,nearest_boundary_type,geofence_caution,inside_mpa,mpa_distance_km,mpa_caution,distance_km,travel_time_hr,fuel_l_proxy
0,PFZ0001,ANDAMAN,13.170278,92.640278,20.0,15.0,20.0,NaN,NaN,NaN,7.999635,Straight baseline,0,0,1053.068365,0,20.0,1.111111,8.888889
1,PFZ0002,ANDAMAN,11.760000,92.946944,16.0,97.0,102.0,NaN,NaN,NaN,66.300979,Straight baseline,0,0,1213.398008,0,16.0,0.888889,7.111111
2,PFZ0003,ANDAMAN,11.745000,93.004444,8.0,67.0,75.0,NaN,NaN,NaN,72.801917,Straight baseline,0,0,1215.136296,0,8.0,0.444444,3.555556
3,PFZ0004,ANDAMAN,11.694444,93.021944,27.0,226.0,249.0,NaN,NaN,NaN,77.071736,Straight baseline,0,0,1220.900345,0,27.0,1.500000,12.000000
4,PFZ0005,ANDAMAN,11.632500,93.021111,33.0,254.0,230.0,NaN,NaN,NaN,80.073879,Straight baseline,0,0,1227.940194,0,33.0,1.833333,14.666667


In [4]:
# STEP 4 — Prepare Route Safety / Constraint Information

safety_cols = [
    "water_depth_m",
    "geofence_caution",
    "inside_mpa",
    "mpa_distance_km",
    "mpa_caution",
    "eez_boundary_distance_km"
]

# Create a separate dataframe for route-safety information
route_safety = routing_data[
    [col for col in safety_cols if col in routing_data.columns]
].copy()

# Convert safety-related numeric fields
for col in [
    "water_depth_m",
    "inside_mpa",
    "mpa_distance_km",
    "eez_boundary_distance_km"
]:
    if col in route_safety.columns:
        route_safety[col] = pd.to_numeric(
            route_safety[col],
            errors="coerce"
        )

# Convert caution fields into clean numeric flags
for col in ["geofence_caution", "mpa_caution"]:
    if col in route_safety.columns:
        route_safety[col] = (
            route_safety[col]
            .astype(str)
            .str.strip()
            .str.upper()
            .map({
                "TRUE": 1,
                "FALSE": 0,
                "YES": 1,
                "NO": 0,
                "1": 1,
                "0": 0
            })
        )

print("✅ Route safety information prepared")

print("\nSafety variables:")
print(route_safety.columns.tolist())

print("\nMissing values:")
print(route_safety.isnull().sum())

print("\nSafety data sample:")
display(route_safety.head())

✅ Route safety information prepared

Safety variables:
['water_depth_m', 'geofence_caution', 'inside_mpa', 'mpa_distance_km', 'mpa_caution', 'eez_boundary_distance_km']

Missing values:
water_depth_m               0
geofence_caution            0
inside_mpa                  0
mpa_distance_km             0
mpa_caution                 0
eez_boundary_distance_km    0
dtype: int64

Safety data sample:


,water_depth_m,geofence_caution,inside_mpa,mpa_distance_km,mpa_caution,eez_boundary_distance_km
0,20.0,0,0,1053.068365,0,7.999635
1,102.0,0,0,1213.398008,0,66.300979
2,75.0,0,0,1215.136296,0,72.801917
3,249.0,0,0,1220.900345,0,77.071736
4,230.0,0,0,1227.940194,0,80.073879


In [5]:
# STEP 5 — Prepare Distance, Travel Time and Fuel

route_cost_data = routing_data[
    [
        "zone_id",
        "latitude",
        "longitude",
        "distance_km",
        "travel_time_hr",
        "fuel_l_proxy"
    ]
].copy()

# Convert routing cost variables to numeric
for col in ["distance_km", "travel_time_hr", "fuel_l_proxy"]:
    route_cost_data[col] = pd.to_numeric(
        route_cost_data[col],
        errors="coerce"
    )

# Check validity
print("✅ Distance / Time / Fuel data prepared")

print("\nMissing values:")
print(route_cost_data.isnull().sum())

print("\nBasic statistics:")
display(
    route_cost_data[
        ["distance_km", "travel_time_hr", "fuel_l_proxy"]
    ].describe()
)

print("\nSample:")
display(route_cost_data.head())

✅ Distance / Time / Fuel data prepared

Missing values:
zone_id           0
latitude          0
longitude         0
distance_km       0
travel_time_hr    0
fuel_l_proxy      0
dtype: int64

Basic statistics:


,distance_km,travel_time_hr,fuel_l_proxy
count,2037.000000,2037.000000,2037.000000
mean,218.027938,12.112663,96.901306
std,147.399515,8.188862,65.510896
min,1.455515,0.080862,0.646896
25%,71.000000,3.944444,31.555556
50%,210.456495,11.692028,93.536220
75%,342.585106,19.032506,152.260047
max,499.526461,27.751470,222.011760



Sample:


,zone_id,latitude,longitude,distance_km,travel_time_hr,fuel_l_proxy
0,PFZ0001,13.170278,92.640278,20.0,1.111111,8.888889
1,PFZ0002,11.760000,92.946944,16.0,0.888889,7.111111
2,PFZ0003,11.745000,93.004444,8.0,0.444444,3.555556
3,PFZ0004,11.694444,93.021944,27.0,1.500000,12.000000
4,PFZ0005,11.632500,93.021111,33.0,1.833333,14.666667


In [6]:
# STEP 6 — Inspect Route Score Reference

score_reference = route_df[
    [
        "zone_id",
        "distance_km",
        "travel_time_hr",
        "fuel_l_proxy",
        "water_depth_m",
        "geofence_caution",
        "inside_mpa",
        "mpa_distance_km",
        "mpa_caution",
        "route_score",
        "routing_class"
    ]
].copy()

print("✅ Route score reference prepared")

print("\nRoute score statistics:")
display(score_reference["route_score"].describe())

print("\nRouting class distribution:")
print(score_reference["routing_class"].value_counts(dropna=False))

print("\nCorrelation with routing variables:")
corr_cols = [
    "distance_km",
    "travel_time_hr",
    "fuel_l_proxy",
    "water_depth_m",
    "mpa_distance_km",
    "route_score"
]

display(score_reference[corr_cols].corr(numeric_only=True)["route_score"].sort_values())

print("\nSample reference rows:")
display(score_reference.head(10))

✅ Route score reference prepared

Route score statistics:


,route_score
count,1542.000000
mean,0.946924
std,0.060050
min,0.508910
25%,0.939034
50%,0.964494
75%,0.979920
max,0.999116



Routing class distribution:
routing_class
GOOD        1520
NaN          495
MODERATE      22
Name: count, dtype: int64

Correlation with routing variables:


,route_score
travel_time_hr,0.072004
fuel_l_proxy,0.072004
distance_km,0.072004
mpa_distance_km,0.096192
water_depth_m,0.253451
route_score,1.000000



Sample reference rows:


,zone_id,distance_km,travel_time_hr,fuel_l_proxy,water_depth_m,geofence_caution,inside_mpa,mpa_distance_km,mpa_caution,route_score,routing_class
0,PFZ0001,20.0,1.111111,8.888889,20.0,0,0,1053.068365,0,NaN,NaN
1,PFZ0002,16.0,0.888889,7.111111,102.0,0,0,1213.398008,0,NaN,NaN
2,PFZ0003,8.0,0.444444,3.555556,75.0,0,0,1215.136296,0,NaN,NaN
3,PFZ0004,27.0,1.500000,12.000000,249.0,0,0,1220.900345,0,NaN,NaN
4,PFZ0005,33.0,1.833333,14.666667,230.0,0,0,1227.940194,0,NaN,NaN
5,PFZ0006,34.0,1.888889,15.111111,453.0,0,0,1233.027342,0,0.985271,GOOD
6,PFZ0007,29.0,1.611111,12.888889,499.0,0,0,1235.717176,0,0.985271,GOOD
7,PFZ0008,30.0,1.666667,13.333333,468.0,0,0,1238.920167,0,0.985271,GOOD
8,PFZ0009,27.0,1.500000,12.000000,601.0,0,0,1242.743572,0,0.985271,GOOD
9,PFZ0010,45.0,2.500000,20.000000,1149.0,0,0,1258.577063,0,0.982371,GOOD


In [7]:
# STEP 7 — Inspect Route Score and Routing Class Mapping

valid_scores = route_df[
    route_df["route_score"].notna() &
    route_df["routing_class"].notna()
].copy()

print("✅ Valid scored routes:", len(valid_scores))

print("\nRoute score range by routing class:")
display(
    valid_scores.groupby("routing_class")["route_score"]
    .agg(["count", "min", "max", "mean", "median"])
)

print("\nSorted score samples:")
display(
    valid_scores[
        ["zone_id", "route_score", "routing_class"]
    ]
    .sort_values("route_score")
    .head(20)
)

print("\nHighest score samples:")
display(
    valid_scores[
        ["zone_id", "route_score", "routing_class"]
    ]
    .sort_values("route_score", ascending=False)
    .head(20)
)

print("\nMissing route score/class rows:")
missing_route = route_df[
    route_df["route_score"].isna() |
    route_df["routing_class"].isna()
]

print("Rows:", len(missing_route))
display(
    missing_route[
        [
            "zone_id",
            "distance_km",
            "travel_time_hr",
            "fuel_l_proxy",
            "water_depth_m",
            "geofence_caution",
            "inside_mpa",
            "mpa_caution"
        ]
    ].head(10)
)

✅ Valid scored routes: 1542

Route score range by routing class:


,count,min,max,mean,median
routing_class,,,,,
GOOD,1520,0.718493,0.999116,0.951550,0.965156
MODERATE,22,0.508910,0.695283,0.627289,0.631783



Sorted score samples:


,zone_id,route_score,routing_class
1179,GRID_0843,0.508910,MODERATE
1452,GRID_1116,0.511041,MODERATE
1345,GRID_1009,0.576277,MODERATE
905,GRID_0569,0.579817,MODERATE
1038,GRID_0702,0.585942,MODERATE
1804,GRID_1468,0.609944,MODERATE
176,PFZ0177,0.611472,MODERATE
1756,GRID_1420,0.611715,MODERATE
1821,GRID_1485,0.615285,MODERATE
174,PFZ0175,0.627028,MODERATE



Highest score samples:


,zone_id,route_score,routing_class
360,GRID_0024,0.999116,GOOD
1850,GRID_1514,0.998992,GOOD
1959,GRID_1623,0.998399,GOOD
239,PFZ0240,0.998370,GOOD
243,PFZ0244,0.998370,GOOD
242,PFZ0243,0.998370,GOOD
241,PFZ0242,0.998370,GOOD
240,PFZ0241,0.998370,GOOD
1427,GRID_1091,0.998210,GOOD
60,PFZ0061,0.997931,GOOD



Missing route score/class rows:
Rows: 495


,zone_id,distance_km,travel_time_hr,fuel_l_proxy,water_depth_m,geofence_caution,inside_mpa,mpa_caution
0,PFZ0001,20.0,1.111111,8.888889,20.0,0,0,0
1,PFZ0002,16.0,0.888889,7.111111,102.0,0,0,0
2,PFZ0003,8.0,0.444444,3.555556,75.0,0,0,0
3,PFZ0004,27.0,1.500000,12.000000,249.0,0,0,0
4,PFZ0005,33.0,1.833333,14.666667,230.0,0,0,0
256,PFZ0257,26.0,1.444444,11.555556,202.0,0,0,0
259,PFZ0260,23.0,1.277778,10.222222,144.0,0,0,0
262,PFZ0263,25.0,1.388889,11.111111,139.0,0,0,0
263,PFZ0264,28.0,1.555556,12.444444,127.0,0,0,0
264,PFZ0265,26.0,1.444444,11.555556,84.0,0,0,0


In [8]:
# STEP 8 — Build Route Score Training Dataset

# Features used by the Route Agent
route_input_features = [
    "distance_km",
    "travel_time_hr",
    "fuel_l_proxy",
    "water_depth_m",
    "geofence_caution",
    "inside_mpa",
    "mpa_distance_km",
    "mpa_caution",
    "eez_boundary_distance_km",
    "current_u_ms",
    "current_v_ms",
    "current_speed_ms"
]

# Keep only columns that actually exist
route_input_features = [
    col for col in route_input_features
    if col in route_df.columns
]

# Rows where the known route score is available
train_df = route_df[
    route_df["route_score"].notna()
].copy()

# Training inputs
X = train_df[route_input_features].copy()

# Target = route score
y = train_df["route_score"].copy()

# Convert all inputs to numeric
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Safety check
print("✅ Route training dataset created")

print("Training rows:", len(X))
print("Input features:", X.shape[1])

print("\nFeatures used:")
print(X.columns.tolist())

print("\nTarget:")
print("route_score")

print("\nMissing values in training inputs:")
print(X.isnull().sum())

print("\nTarget range:")
print("Min:", y.min())
print("Max:", y.max())


✅ Route training dataset created
Training rows: 1542
Input features: 12

Features used:
['distance_km', 'travel_time_hr', 'fuel_l_proxy', 'water_depth_m', 'geofence_caution', 'inside_mpa', 'mpa_distance_km', 'mpa_caution', 'eez_boundary_distance_km', 'current_u_ms', 'current_v_ms', 'current_speed_ms']

Target:
route_score

Missing values in training inputs:
distance_km                 0
travel_time_hr              0
fuel_l_proxy                0
water_depth_m               0
geofence_caution            0
inside_mpa                  0
mpa_distance_km             0
mpa_caution                 0
eez_boundary_distance_km    0
current_u_ms                0
current_v_ms                0
current_speed_ms            0
dtype: int64

Target range:
Min: 0.5089100333333334
Max: 0.99911611655


In [9]:
# STEP 9 — Train/Test Split + Route Score Model

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Split the known-score data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("✅ Train/Test split completed")
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

# Handle missing routing values
imputer = SimpleImputer(strategy="median")

# Route score prediction model
model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    max_depth=12,
    min_samples_leaf=2,
    n_jobs=-1
)

# Pipeline
route_model = Pipeline([
    ("imputer", imputer),
    ("model", model)
])

# Train
route_model.fit(X_train, y_train)

# Predict
y_pred = route_model.predict(X_test)

# Keep score inside 0–1
y_pred = np.clip(y_pred, 0, 1)

# Evaluation
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n✅ Route Score model trained")

print("\nModel performance:")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

✅ Train/Test split completed
Training rows: 1233
Testing rows: 309

✅ Route Score model trained

Model performance:
MAE  : 0.0017
RMSE : 0.0085
R²   : 0.9759


In [10]:
# STEP 10 — Route Score to Routing Class

# Find the observed boundary between MODERATE and GOOD
class_ranges = (
    valid_scores.groupby("routing_class")["route_score"]
    .agg(["min", "max"])
)

print("Observed routing class ranges:")
display(class_ranges)

# Use the midpoint between the highest MODERATE score
# and the lowest GOOD score as the classification threshold.
moderate_max = class_ranges.loc["MODERATE", "max"]
good_min = class_ranges.loc["GOOD", "min"]

ROUTE_SCORE_THRESHOLD = (moderate_max + good_min) / 2

print(f"\n✅ Route score threshold: {ROUTE_SCORE_THRESHOLD:.4f}")


def classify_route_score(score):
    """
    Convert a route score into the dataset's routing class.
    """
    if score >= ROUTE_SCORE_THRESHOLD:
        return "GOOD"
    return "MODERATE"


# Create test prediction table
prediction_results = X_test.copy()

prediction_results["actual_route_score"] = y_test.values
prediction_results["predicted_route_score"] = y_pred

prediction_results["predicted_routing_class"] = (
    prediction_results["predicted_route_score"]
    .apply(classify_route_score)
)

print("\n✅ Prediction output created")

display(
    prediction_results[
        [
            "actual_route_score",
            "predicted_route_score",
            "predicted_routing_class"
        ]
    ].head(15)
)

Observed routing class ranges:


,min,max
routing_class,,
GOOD,0.718493,0.999116
MODERATE,0.508910,0.695283



✅ Route score threshold: 0.7069

✅ Prediction output created


,actual_route_score,predicted_route_score,predicted_routing_class
1965,0.958927,0.958719,GOOD
1500,0.952708,0.952700,GOOD
1647,0.942450,0.942485,GOOD
698,0.831490,0.810596,GOOD
1425,0.978249,0.978322,GOOD
727,0.996813,0.996828,GOOD
1996,0.913257,0.913336,GOOD
1737,0.957309,0.957315,GOOD
1573,0.996418,0.996590,GOOD
1735,0.961194,0.961196,GOOD


In [13]:
# STEP 11 — Route Agent Prediction Function

def predict_route(zone_id):
    """
    Generate the Route Agent output for a given zone_id.
    """

    # Find the requested zone
    row = route_df[route_df["zone_id"] == zone_id]

    if row.empty:
        return {
            "zone_id": zone_id,
            "status": "ERROR",
            "message": "Zone ID not found"
        }

    row = row.iloc[0]

    # Prepare model input
    input_data = pd.DataFrame([{
        feature: pd.to_numeric(row[feature], errors="coerce")
        for feature in route_input_features
    }])

    # Predict route score
    predicted_score = float(
        np.clip(
            route_model.predict(input_data)[0],
            0,
            1
        )
    )

    # Convert score to routing class
    routing_class = classify_route_score(predicted_score)

    # Build Route Agent output
    result = {
        "zone_id": zone_id,
        "route_score": round(predicted_score, 4),
        "distance_km": (
            None if pd.isna(row["distance_km"])
            else round(float(row["distance_km"]), 4)
        ),
        "travel_time_hr": (
            None if pd.isna(row["travel_time_hr"])
            else round(float(row["travel_time_hr"]), 4)
        ),
        "fuel_l": (
            None if pd.isna(row["fuel_l_proxy"])
            else round(float(row["fuel_l_proxy"]), 4)
        ),
        "routing_class": routing_class,
        "status": "SUCCESS"
    }

    return result


In [14]:
# STEP 11 TEST

test_zone = route_df["zone_id"].iloc[0]

route_result = predict_route(test_zone)

print("✅ Route Agent prediction")
print(route_result)

✅ Route Agent prediction
{'zone_id': 'PFZ0001', 'route_score': 0.7123, 'distance_km': 20.0, 'travel_time_hr': 1.1111, 'fuel_l': 8.8889, 'routing_class': 'GOOD', 'status': 'SUCCESS'}


In [15]:
# STEP 12 — Generate Route Results for Multiple Zones

# Select first 10 zones for testing
test_zones = route_df["zone_id"].head(10).tolist()

route_results = []

for zone_id in test_zones:
    result = predict_route(zone_id)

    if result.get("status") == "SUCCESS":
        route_results.append(result)

# Convert results to DataFrame
route_results_df = pd.DataFrame(route_results)

# Rank by Route Score
route_results_df = route_results_df.sort_values(
    "route_score",
    ascending=False
).reset_index(drop=True)

route_results_df["route_rank"] = (
    route_results_df.index + 1
)

print("✅ Route Agent ranking generated")

display(
    route_results_df[
        [
            "route_rank",
            "zone_id",
            "route_score",
            "distance_km",
            "travel_time_hr",
            "fuel_l",
            "routing_class"
        ]
    ]
)

✅ Route Agent ranking generated


,route_rank,zone_id,route_score,distance_km,travel_time_hr,fuel_l,routing_class
0,1,PFZ0009,0.9853,27.0,1.5000,12.0000,GOOD
1,2,PFZ0006,0.9853,34.0,1.8889,15.1111,GOOD
2,3,PFZ0008,0.9853,30.0,1.6667,13.3333,GOOD
3,4,PFZ0007,0.9853,29.0,1.6111,12.8889,GOOD
4,5,PFZ0010,0.9824,45.0,2.5000,20.0000,GOOD
5,6,PFZ0002,0.9694,16.0,0.8889,7.1111,GOOD
6,7,PFZ0005,0.9694,33.0,1.8333,14.6667,GOOD
7,8,PFZ0004,0.9694,27.0,1.5000,12.0000,GOOD
8,9,PFZ0003,0.9692,8.0,0.4444,3.5556,GOOD
9,10,PFZ0001,0.7123,20.0,1.1111,8.8889,GOOD


In [16]:
# STEP 13 — Save M3 Route Agent Handover Files

import joblib
import os
import textwrap

# Create M3 folder
os.makedirs("M3", exist_ok=True)

# Save trained route model
joblib.dump(
    route_model,
    "M3/route_model.pkl"
)

# Generate standalone prediction function
predict_code = f'''
import joblib
import numpy as np
import pandas as pd

# Load trained Route Agent model
route_model = joblib.load("route_model.pkl")

# Route score classification threshold
ROUTE_SCORE_THRESHOLD = {ROUTE_SCORE_THRESHOLD}

# Features expected by the Route Agent
ROUTE_INPUT_FEATURES = {route_input_features!r}


def classify_route_score(score):
    """Convert Route Score into routing class."""
    if score >= ROUTE_SCORE_THRESHOLD:
        return "GOOD"
    return "MODERATE"


def predict_route(input_data):
    """
    Route Agent prediction interface.

    input_data must contain the routing features required by the model.
    """

    # Convert input dictionary to DataFrame
    X_input = pd.DataFrame([input_data])

    # Make sure all required features are present
    for feature in ROUTE_INPUT_FEATURES:
        if feature not in X_input.columns:
            X_input[feature] = np.nan

    X_input = X_input[ROUTE_INPUT_FEATURES]

    # Convert values to numeric
    for feature in ROUTE_INPUT_FEATURES:
        X_input[feature] = pd.to_numeric(
            X_input[feature],
            errors="coerce"
        )

    # Predict Route Score
    route_score = float(
        np.clip(
            route_model.predict(X_input)[0],
            0,
            1
        )
    )

    result = {{
        "zone_id": input_data.get("zone_id"),
        "route_score": round(route_score, 4),
        "distance_km": input_data.get("distance_km"),
        "travel_time_hr": input_data.get("travel_time_hr"),
        "fuel_l": input_data.get("fuel_l_proxy"),
        "routing_class": classify_route_score(route_score),
        "status": "SUCCESS"
    }}

    return result
'''

with open("M3/predict.py", "w") as f:
    f.write(textwrap.dedent(predict_code))

print("✅ M3 handover files created")

print("\nFiles:")
print("M3/route_model.pkl")
print("M3/predict.py")

✅ M3 handover files created

Files:
M3/route_model.pkl
M3/predict.py


In [21]:
# Reload corrected M3 prediction module

import sys
import importlib

sys.path.insert(0, "M3")

import predict
importlib.reload(predict)

from predict import predict_route

print("✅ predict.py loaded successfully")

✅ predict.py loaded successfully


In [20]:
import sys

sys.path.insert(0, "M3")

from predict import predict_route

sample_input = {
    "zone_id": route_df["zone_id"].iloc[0],
    "distance_km": route_df["distance_km"].iloc[0],
    "travel_time_hr": route_df["travel_time_hr"].iloc[0],
    "fuel_l_proxy": route_df["fuel_l_proxy"].iloc[0],
    "water_depth_m": route_df["water_depth_m"].iloc[0],
    "geofence_caution": route_df["geofence_caution"].iloc[0],
    "inside_mpa": route_df["inside_mpa"].iloc[0],
    "mpa_distance_km": route_df["mpa_distance_km"].iloc[0],
    "mpa_caution": route_df["mpa_caution"].iloc[0],
    "eez_boundary_distance_km": route_df["eez_boundary_distance_km"].iloc[0],
    "current_u_ms": route_df["current_u_ms"].iloc[0],
    "current_v_ms": route_df["current_v_ms"].iloc[0],
    "current_speed_ms": route_df["current_speed_ms"].iloc[0]
}

test_result = predict_route(sample_input)

print("✅ M3 handover interface test")
print(test_result)

✅ M3 handover interface test
{'zone_id': 'PFZ0001', 'route_score': 0.7123, 'distance_km': np.float64(20.0), 'travel_time_hr': np.float64(1.1111111111111112), 'fuel_l': np.float64(8.88888888888889), 'routing_class': 'GOOD', 'status': 'SUCCESS'}


In [22]:
# STEP 15 — Final Route Agent JSON Output Test

import json

test_zone = route_df["zone_id"].iloc[0]

row = route_df[
    route_df["zone_id"] == test_zone
].iloc[0]

sample_input = {
    "zone_id": test_zone,

    "distance_km": row["distance_km"],
    "travel_time_hr": row["travel_time_hr"],
    "fuel_l_proxy": row["fuel_l_proxy"],

    "water_depth_m": row["water_depth_m"],
    "geofence_caution": row["geofence_caution"],
    "inside_mpa": row["inside_mpa"],
    "mpa_distance_km": row["mpa_distance_km"],
    "mpa_caution": row["mpa_caution"],
    "eez_boundary_distance_km": row["eez_boundary_distance_km"],

    "current_u_ms": row["current_u_ms"],
    "current_v_ms": row["current_v_ms"],
    "current_speed_ms": row["current_speed_ms"]
}

# Call M3 Route Agent
result = predict_route(sample_input)

print("✅ M3 ROUTE AGENT OUTPUT\n")
print(
    json.dumps(
        result,
        indent=4,
        default=str
    )
)

✅ M3 ROUTE AGENT OUTPUT

{
    "zone_id": "PFZ0001",
    "route_score": 0.7123,
    "distance_km": 20.0,
    "travel_time_hr": 1.1111111111111112,
    "fuel_l": 8.88888888888889,
    "routing_class": "GOOD",
    "status": "SUCCESS"
}


In [23]:
# STEP 16 — Validate M3 Route Agent

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Validation on known route-score rows
# ---------------------------------------------------------

validation_df = train_df.copy()

X_validation = validation_df[route_input_features].copy()

for col in X_validation.columns:
    X_validation[col] = pd.to_numeric(
        X_validation[col],
        errors="coerce"
    )

actual_scores = validation_df["route_score"].values

predicted_scores = route_model.predict(X_validation)
predicted_scores = np.clip(predicted_scores, 0, 1)

mae = mean_absolute_error(actual_scores, predicted_scores)
rmse = np.sqrt(mean_squared_error(actual_scores, predicted_scores))
r2 = r2_score(actual_scores, predicted_scores)

print("✅ M3 validation completed")

print("\nValidation Metrics")
print("------------------")
print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")


# ---------------------------------------------------------
# 2. Generate Route Agent predictions for missing-score rows
# ---------------------------------------------------------

unscored_df = route_df[
    route_df["route_score"].isna()
].copy()

X_unscored = unscored_df[route_input_features].copy()

for col in X_unscored.columns:
    X_unscored[col] = pd.to_numeric(
        X_unscored[col],
        errors="coerce"
    )

unscored_predictions = route_model.predict(X_unscored)
unscored_predictions = np.clip(unscored_predictions, 0, 1)

unscored_df["predicted_route_score"] = unscored_predictions
unscored_df["predicted_routing_class"] = (
    unscored_df["predicted_route_score"]
    .apply(classify_route_score)
)

print("\n✅ Predictions generated for previously unscored routes")
print("Rows predicted:", len(unscored_df))

display(
    unscored_df[
        [
            "zone_id",
            "distance_km",
            "travel_time_hr",
            "fuel_l_proxy",
            "predicted_route_score",
            "predicted_routing_class"
        ]
    ].head(20)
)

✅ M3 validation completed

Validation Metrics
------------------
MAE : 0.0010
RMSE: 0.0055
R²  : 0.9917

✅ Predictions generated for previously unscored routes
Rows predicted: 495


,zone_id,distance_km,travel_time_hr,fuel_l_proxy,predicted_route_score,predicted_routing_class
0,PFZ0001,20.0,1.111111,8.888889,0.712279,GOOD
1,PFZ0002,16.0,0.888889,7.111111,0.969352,GOOD
2,PFZ0003,8.0,0.444444,3.555556,0.969181,GOOD
3,PFZ0004,27.0,1.500000,12.000000,0.969360,GOOD
4,PFZ0005,33.0,1.833333,14.666667,0.969359,GOOD
256,PFZ0257,26.0,1.444444,11.555556,0.969284,GOOD
259,PFZ0260,23.0,1.277778,10.222222,0.969284,GOOD
262,PFZ0263,25.0,1.388889,11.111111,0.969276,GOOD
263,PFZ0264,28.0,1.555556,12.444444,0.969276,GOOD
264,PFZ0265,26.0,1.444444,11.555556,0.969276,GOOD


In [24]:
# STEP 17 — Final M3 Handover Verification

import os

model_path = "M3/route_model.pkl"
predict_path = "M3/predict.py"

print("M3 HANDOVER")
print("-----------")

print(
    "✅ route_model.pkl :",
    os.path.exists(model_path),
    f"({os.path.getsize(model_path) / 1024:.2f} KB)"
)

print(
    "✅ predict.py      :",
    os.path.exists(predict_path),
    f"({os.path.getsize(predict_path) / 1024:.2f} KB)"
)

print("\nFolder contents:")

for file in os.listdir("M3"):
    print(" -", file)

M3 HANDOVER
-----------
✅ route_model.pkl : True (10279.21 KB)
✅ predict.py      : True (1.86 KB)

Folder contents:
 - route_model.pkl
 - predict.py
 - __pycache__


In [25]:
# STEP 18 — Final M3 JSON Contract Test

import json

# Select one test zone
test_zone = route_df["zone_id"].iloc[0]

row = route_df[
    route_df["zone_id"] == test_zone
].iloc[0]

# Prepare Route Agent input
route_input = {
    "zone_id": str(test_zone),

    "distance_km": row["distance_km"],
    "travel_time_hr": row["travel_time_hr"],
    "fuel_l_proxy": row["fuel_l_proxy"],

    "water_depth_m": row["water_depth_m"],
    "geofence_caution": row["geofence_caution"],
    "inside_mpa": row["inside_mpa"],
    "mpa_distance_km": row["mpa_distance_km"],
    "mpa_caution": row["mpa_caution"],
    "eez_boundary_distance_km": row["eez_boundary_distance_km"],

    "current_u_ms": row["current_u_ms"],
    "current_v_ms": row["current_v_ms"],
    "current_speed_ms": row["current_speed_ms"]
}

# Run Route Agent
route_output = predict_route(route_input)

# Final JSON
final_route_json = {
    "route": {
        "zone_id": route_output["zone_id"],
        "route_score": route_output["route_score"],
        "distance_km": route_output["distance_km"],
        "travel_time_hr": route_output["travel_time_hr"],
        "fuel_l": route_output["fuel_l"]
    }
}

print("✅ FINAL M3 ROUTE AGENT OUTPUT\n")
print(json.dumps(final_route_json, indent=4))

✅ FINAL M3 ROUTE AGENT OUTPUT

{
    "route": {
        "zone_id": "PFZ0001",
        "route_score": 0.7123,
        "distance_km": 20.0,
        "travel_time_hr": 1.1111111111111112,
        "fuel_l": 8.88888888888889
    }
}


In [26]:
# STEP 19 — Create Final M3 Handover Package

import os
import shutil

# Final package folder
os.makedirs("M3_FINAL", exist_ok=True)

# Copy the two handover files
shutil.copy(
    "M3/route_model.pkl",
    "M3_FINAL/route_model.pkl"
)

shutil.copy(
    "M3/predict.py",
    "M3_FINAL/predict.py"
)

print("✅ M3 final handover package created")

print("\nContents:")
for file in os.listdir("M3_FINAL"):
    print(" -", file)

✅ M3 final handover package created

Contents:
 - route_model.pkl
 - predict.py
